# V-Scan Object completion

In [ ]:
import os
import trimesh
import torch
from context import drm, inference

In [ ]:
from inference import Hunyuan3DOmniSiTFlowMatchingPipeline,FloaterRemover, DegenerateFaceRemover
# Load the pipeline
pipeline = Hunyuan3DOmniSiTFlowMatchingPipeline.from_pretrained(
    "tencent/Hunyuan3D-Omni", 
    fast_decode = True
)

In [ ]:
import os
import glob
import trimesh

# Point this at a single scene folder (like below) or higher up
# (e.g. .../selections/bedroom or .../selections) to process multiple
# scenes/categories in one go — the glob below is recursive either way.
SELECTIONS_ROOT = "/home/jelle-vermandere/Documents/Github/data/selections"

SHOW_MESHES = False  # set True to pop open a viewer window after each completion

# Every object subfolder has a scan.ply -> use that to discover all objects
object_dirs = sorted(
    os.path.dirname(p)
    for p in glob.glob(os.path.join(SELECTIONS_ROOT, "**", "scan.ply"), recursive=True)
)
print(f"Found {len(object_dirs)} object(s) to complete")

results_log = {"ok": [], "skipped": [], "error": []}

for object_dir in object_dirs:
    object_name = os.path.basename(object_dir)
    scan_path = os.path.join(object_dir, "scan.ply")

    crop_candidates = glob.glob(os.path.join(object_dir, "crop.*"))
    if not crop_candidates:
        print(f"  [skip] {object_name}: no crop image found")
        results_log["skipped"].append(object_name)
        continue
    imagePath = crop_candidates[0]

    try:
        pcd = trimesh.load(scan_path, process=False)

        partial_pcd = inference.normalize_mesh(pcd, scale=0.98)
        surface = partial_pcd.vertices
        surface = torch.FloatTensor(surface).unsqueeze(0)
        surface = surface.to(pipeline.device).to(pipeline.dtype)

        result = pipeline(
            image=imagePath,
            point=surface,
            num_inference_steps=20,
            octree_resolution=256,
            mc_level=0,
            guidance_scale=4.5,
            generator=torch.Generator('cuda').manual_seed(1234),
        )

        mesh = result['shapes'][0][0]
        sampled_point = result['sampled_point'][0]
        mesh = FloaterRemover()(mesh)
        mesh = DegenerateFaceRemover()(mesh)

        # save the completion right alongside crop.png / gt.ply / scan.ply
        out_path = os.path.join(object_dir, "completion.glb")
        mesh.export(out_path)

        if SHOW_MESHES:
            mesh.show()

        print(f"  [ok] {object_name} -> {out_path}")
        results_log["ok"].append(object_name)

    except Exception as e:
        print(f"  [error] {object_name}: {e}")
        results_log["error"].append((object_name, str(e)))

print()
print(f"Done. ok={len(results_log['ok'])} skipped={len(results_log['skipped'])} error={len(results_log['error'])}")

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("/home/jelle-vermandere/Documents/Github/Hunyuan3D-2"))
import hy3dgen

from hy3dgen.texgen import Hunyuan3DPaintPipeline
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.rembg import BackgroundRemover
import trimesh
from PIL import Image

pipeline = Hunyuan3DPaintPipeline.from_pretrained(
    'tencent/Hunyuan3D-2',
    subfolder='hunyuan3d-paint-v2-0-turbo'
)

In [ ]:
import os
import glob
import trimesh
from PIL import Image

# Point this at a single scene folder (like below) or higher up
# (e.g. .../selections/bedroom or .../selections) to process multiple
# scenes/categories in one go — the glob below is recursive either way.
SELECTIONS_ROOT = "/home/jelle-vermandere/Documents/Github/data/texture_selection"

SHOW_MESHES = False  # set True to pop open a viewer window after each completion

# Every object subfolder has a scan.ply -> use that to discover all objects
object_dirs = sorted(
    os.path.dirname(p)
    for p in glob.glob(os.path.join(SELECTIONS_ROOT, "**", "completion.glb"), recursive=True)
)
print(f"Found {len(object_dirs)} object(s) to paint")

results_log = {"ok": [], "skipped": [], "error": []}

for object_dir in object_dirs:
    object_name = os.path.basename(object_dir)
    mesh_path = os.path.join(object_dir, "completion.glb")

    crop_candidates = glob.glob(os.path.join(object_dir, "crop.*"))
    if not crop_candidates:
        print(f"  [skip] {object_name}: no crop image found")
        results_log["skipped"].append(object_name)
        continue
    imagePath = crop_candidates[0]

    try:
        image = Image.open(imagePath).convert("RGBA")
        mesh = trimesh.load(mesh_path)

        if image.mode == 'RGB':
            rembg = BackgroundRemover()
            image = rembg(image)
        textured_mesh = pipeline(mesh, image=image)

        # save the completion right alongside crop.png / gt.ply / scan.ply
        out_path = os.path.join(object_dir, "completion_textured.glb")
        textured_mesh.export(out_path)

        if SHOW_MESHES:
            mesh.show()

        print(f"  [ok] {object_name} -> {out_path}")
        results_log["ok"].append(object_name)

    except Exception as e:
        print(f"  [error] {object_name}: {e}")
        results_log["error"].append((object_name, str(e)))

print()
print(f"Done. ok={len(results_log['ok'])} skipped={len(results_log['skipped'])} error={len(results_log['error'])}")

In [ ]:
# =============================================================================
# Texture/appearance evaluation: completed+textured objects vs GT assets
#
# Folder layout expected:
#   TEXTURE_SELECTION_ROOT/<category>/<object_folder>/completion_textured.glb
#   GT_ROOT/<dataset>/<category>/<object_id>.glb   (searched recursively)
#
# <object_folder> is matched to a GT file by stripping a trailing
# "(Clone)"-style suffix to get an object_id, then searching GT_ROOT for a
# glb whose filename stem either *is* that id (ShapeNet-style, e.g.
# '1e46876da968e5de233e9dd8288dd718.glb') or *contains* that id in a
# trailing parenthetical group (IKEA-style, e.g. 'KALLAX  LAGKAPTEN desk
# combination - whitewhite stained oak effect (49481682).glb').
#
# Metrics (averaged over a turntable of rendered views): PSNR, SSIM, LPIPS.
#
# Method: since the GT asset and the completed mesh have different
# topology/UVs and unknown relative scale/pose, texture fidelity is compared
# via rendering rather than direct UV-pixel sampling:
#   1. independently normalize each mesh to a unit sphere
#   2. sanitize materials (non-metallic/matte/white baseColorFactor) so only
#      the texture image drives appearance, not default PBR shading
#   3. register completion -> GT via multi-start ICP (24 cube-symmetry
#      rotation candidates, refined) - resolves unknown canonical pose
#      conventions between the two assets
#   4. render both from the same turntable viewpoints, compute PSNR/SSIM/
#      LPIPS per view, average per object
#
# RENDERING BACKEND: set PYOPENGL_PLATFORM *before* importing pyrender -
# it can't be switched later in the same kernel session. 'egl' needs a
# GPU/EGL; 'osmesa' is pure software (needs `apt install libosmesa6`) but
# works without a GPU. If you hit a `glGenTextures` crash on textured
# meshes, upgrade PyOpenGL: pip install --upgrade PyOpenGL PyOpenGL-accelerate
# (pyrender pins an old PyOpenGL==3.1.0 that's broken with numpy>=2).
# =============================================================================

import os
os.environ["PYOPENGL_PLATFORM"] = "egl"  # or "osmesa" - see note above

import re
import glob
import traceback
from itertools import permutations, product

import numpy as np
import pandas as pd
import trimesh
import open3d as o3d
import pyrender
import torch
import lpips
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# --- config ------------------------------------------------------------------

TEXTURE_SELECTION_ROOT = "/home/jelle-vermandere/Documents/Github/data/texture_selection"
GT_ROOT = "/home/jelle-vermandere/Documents/Github/data/gt"
OUTPUT_DIR = os.path.join(TEXTURE_SELECTION_ROOT, "_eval")

N_VIEWS = 8
RESOLUTION = 512
N_ICP_POINTS = 5000
LPIPS_NET = "alex"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_RENDER_DIR = None  # set to a path to dump GT/pred view images per object for sanity-checking
LIMIT = None  # set to an int to only process the first N objects

os.makedirs(OUTPUT_DIR, exist_ok=True)
if SAVE_RENDER_DIR:
    os.makedirs(SAVE_RENDER_DIR, exist_ok=True)


# --- discovery / matching -----------------------------------------------------

def strip_clone_suffix(name):
    """ '1e46876da968e5de233e9dd8288dd718(Clone)' -> '1e46876da968e5de233e9dd8288dd718' """
    return re.sub(r"\(.*\)\s*$", "", name).strip()


def find_completion_files(texture_selection_root):
    items = []
    for category in sorted(os.listdir(texture_selection_root)):
        cat_dir = os.path.join(texture_selection_root, category)
        if not os.path.isdir(cat_dir):
            continue
        for obj in sorted(os.listdir(cat_dir)):
            obj_dir = os.path.join(cat_dir, obj)
            if not os.path.isdir(obj_dir):
                continue
            completion_path = os.path.join(obj_dir, "completion_textured.glb")
            if not os.path.isfile(completion_path):
                print(f"  [skip] {category}/{obj}: no completion_textured.glb")
                continue
            items.append({"category": category, "object": obj, "completion_path": completion_path})
    return items


def extract_candidate_ids(stem):
    """
    A GT filename's stem can be:
      - a bare id, e.g. '1e46876da968e5de233e9dd8288dd718' (ShapeNet-style)
      - a human-readable name with the id embedded in a trailing
        parenthetical group, e.g.
        'KALLAX  LAGKAPTEN desk combination - whitewhite stained oak effect (49481682)'
        (IKEA-style)
    Returns every plausible id token from the stem: the full stem itself,
    plus the contents of every (...) group found in it.
    """
    candidates = {stem.strip()}
    for m in re.finditer(r"\(([^)]+)\)", stem):
        candidates.add(m.group(1).strip())
    return candidates


def build_gt_index(gt_root):
    """One pass over gt_root, mapping every candidate id -> list of glb paths."""
    index = {}
    for path in glob.glob(os.path.join(gt_root, "**", "*.glb"), recursive=True):
        stem = os.path.splitext(os.path.basename(path))[0]
        for candidate in extract_candidate_ids(stem):
            index.setdefault(candidate, []).append(path)
    return index


def find_gt_glb(gt_index, object_id):
    return sorted(gt_index.get(object_id, []))


# --- mesh loading / material sanitization / normalization --------------------

def load_scene_parts(path):
    """Return a list of trimesh.Trimesh parts (handles single- and multi-mesh glb)."""
    loaded = trimesh.load(path, process=False)
    if isinstance(loaded, trimesh.Scene):
        return [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    elif isinstance(loaded, trimesh.Trimesh):
        return [loaded]
    return []


def sanitize_material(mesh):
    """Force non-metallic / matte / white baseColorFactor so only the texture
    image itself drives appearance (avoids the dark/metallic-default bias)."""
    visual = mesh.visual
    if isinstance(visual, trimesh.visual.texture.TextureVisuals) and visual.material is not None:
        mat = visual.material
        tex_image = getattr(mat, "baseColorTexture", None) or getattr(mat, "image", None)
        if tex_image is not None:
            new_mat = trimesh.visual.material.PBRMaterial(
                baseColorTexture=tex_image,
                baseColorFactor=[255, 255, 255, 255],
                metallicFactor=0.0,
                roughnessFactor=1.0,
            )
            mesh.visual = trimesh.visual.texture.TextureVisuals(uv=visual.uv, material=new_mat)
    return mesh


def normalize_parts(parts):
    """Independently fit a part-list to a unit sphere centered at the origin."""
    all_verts = np.concatenate([p.vertices for p in parts], axis=0)
    center = (all_verts.min(0) + all_verts.max(0)) / 2.0
    radius = np.linalg.norm(all_verts - center, axis=1).max()
    if radius < 1e-12:
        raise ValueError("degenerate mesh, zero radius")
    out = []
    for p in parts:
        q = p.copy()
        q.vertices = (q.vertices - center) / radius
        out.append(q)
    return out


def apply_transform_to_parts(parts, T):
    out = []
    for p in parts:
        q = p.copy()
        q.apply_transform(T)
        out.append(q)
    return out


# --- multi-start rigid registration (resolves unknown canonical pose convention) --

def candidate_rotations():
    """The 24 proper rotations of the cube - every combination of 'which
    world axis the object's up-axis maps to' x '90-degree yaw around it'."""
    mats = []
    for perm in permutations(range(3)):
        for signs in product([1, -1], repeat=3):
            M = np.zeros((3, 3))
            for i, p in enumerate(perm):
                M[i, p] = signs[i]
            if abs(np.linalg.det(M) - 1.0) < 1e-6:
                mats.append(M)
    return mats


def multi_start_icp(source_points, target_points, coarse_corr=0.5, fine_corr=0.08):
    src_pcd = o3d.geometry.PointCloud()
    src_pcd.points = o3d.utility.Vector3dVector(source_points)
    tgt_pcd = o3d.geometry.PointCloud()
    tgt_pcd.points = o3d.utility.Vector3dVector(target_points)

    best, best_score = None, None
    for R in candidate_rotations():
        init = np.eye(4)
        init[:3, :3] = R
        result = o3d.pipelines.registration.registration_icp(
            src_pcd, tgt_pcd, coarse_corr, init,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        )
        # fitness (overlap ratio) can tie across several candidates when
        # coarse_corr is loose relative to shape scale - break ties by
        # preferring lower inlier_rmse, not insertion order.
        score = (result.fitness, -result.inlier_rmse)
        if best is None or score > best_score:
            best, best_score = result, score

    refined = o3d.pipelines.registration.registration_icp(
        src_pcd, tgt_pcd, fine_corr, best.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    )
    return refined.transformation, refined.fitness, refined.inlier_rmse


def register_completion_to_gt(gt_parts, completion_parts, n_icp_points=5000):
    gt_combined = trimesh.util.concatenate(gt_parts)
    pred_combined = trimesh.util.concatenate(completion_parts)

    gt_pts, _ = trimesh.sample.sample_surface(gt_combined, n_icp_points)
    pred_pts, _ = trimesh.sample.sample_surface(pred_combined, n_icp_points)

    T, fitness, rmse = multi_start_icp(pred_pts, gt_pts)
    aligned_parts = apply_transform_to_parts(completion_parts, T)
    return aligned_parts, fitness, rmse


# --- rendering -----------------------------------------------------------------

def render_views(parts, n_views=8, resolution=512, distance=2.6, elevation_deg=15, bg_color=(0.5, 0.5, 0.5)):
    scene = pyrender.Scene(bg_color=list(bg_color) + [1.0], ambient_light=[0.25, 0.25, 0.25])
    for p in parts:
        scene.add(pyrender.Mesh.from_trimesh(p, smooth=False))

    images = []
    r = pyrender.OffscreenRenderer(resolution, resolution)
    try:
        for i in range(n_views):
            az = 2 * np.pi * i / n_views
            el = np.radians(elevation_deg)
            cam_pos = distance * np.array([np.cos(el) * np.sin(az), np.sin(el), np.cos(el) * np.cos(az)])

            forward = -cam_pos / np.linalg.norm(cam_pos)
            world_up = np.array([0.0, 1.0, 0.0])
            right = np.cross(forward, world_up)
            right /= np.linalg.norm(right)
            up = np.cross(right, forward)

            cam_pose = np.eye(4)
            cam_pose[:3, 0] = right
            cam_pose[:3, 1] = up
            cam_pose[:3, 2] = -forward
            cam_pose[:3, 3] = cam_pos

            cam = pyrender.PerspectiveCamera(yfov=np.radians(50))
            cam_node = scene.add(cam, pose=cam_pose)
            light = pyrender.DirectionalLight(intensity=3.0)
            light_node = scene.add(light, pose=cam_pose)

            color, _ = r.render(scene)
            images.append(color)

            scene.remove_node(cam_node)
            scene.remove_node(light_node)
    finally:
        r.delete()

    return images


# --- metrics ---------------------------------------------------------------

def compute_view_metrics(gt_img, pred_img, loss_fn, device):
    gt_f = gt_img.astype(np.float32) / 255.0
    pred_f = pred_img.astype(np.float32) / 255.0

    psnr = peak_signal_noise_ratio(gt_f, pred_f, data_range=1.0)
    ssim = structural_similarity(gt_f, pred_f, channel_axis=2, data_range=1.0)

    def to_lpips_tensor(img_f):
        t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0).float()
        return (t * 2.0 - 1.0).to(device)

    with torch.no_grad():
        lp = float(loss_fn(to_lpips_tensor(gt_f), to_lpips_tensor(pred_f)).item())

    return psnr, ssim, lp


# --- main loop -----------------------------------------------------------------

print("Loading LPIPS model...")
loss_fn = lpips.LPIPS(net=LPIPS_NET).eval().to(DEVICE)

items = find_completion_files(TEXTURE_SELECTION_ROOT)
if LIMIT:
    items = items[:LIMIT]
print(f"\nFound {len(items)} completed object(s)\n")

print("Indexing GT glb files...")
gt_index = build_gt_index(GT_ROOT)
print(f"Indexed {len(gt_index)} candidate id(s) from {GT_ROOT}\n")

rows = []

for item in items:
    tag = f"{item['category']}/{item['object']}"
    row = {"category": item["category"], "object": item["object"], "status": "ok"}

    try:
        object_id = strip_clone_suffix(item["object"])
        gt_matches = find_gt_glb(gt_index, object_id)
        if len(gt_matches) == 0:
            raise FileNotFoundError(f"no GT glb found for object id '{object_id}' under {GT_ROOT}")
        if len(gt_matches) > 1:
            print(f"  [warn] {tag}: {len(gt_matches)} GT matches for '{object_id}', using first: {gt_matches[0]}")
        gt_path = gt_matches[0]
        row["gt_path"] = gt_path

        gt_parts = load_scene_parts(gt_path)
        completion_parts = load_scene_parts(item["completion_path"])
        if not gt_parts or not completion_parts:
            raise ValueError("empty mesh (no valid parts) in GT or completion file")

        gt_parts = [sanitize_material(p) for p in gt_parts]
        completion_parts = [sanitize_material(p) for p in completion_parts]

        gt_parts = normalize_parts(gt_parts)
        completion_parts = normalize_parts(completion_parts)

        aligned_parts, fitness, rmse = register_completion_to_gt(
            gt_parts, completion_parts, n_icp_points=N_ICP_POINTS,
        )
        row["icp_fitness"] = fitness
        row["icp_inlier_rmse"] = rmse

        gt_views = render_views(gt_parts, n_views=N_VIEWS, resolution=RESOLUTION)
        pred_views = render_views(aligned_parts, n_views=N_VIEWS, resolution=RESOLUTION)

        if SAVE_RENDER_DIR:
            safe_name = f"{item['category']}_{item['object']}".replace("/", "_")
            for i, (g, p) in enumerate(zip(gt_views, pred_views)):
                Image.fromarray(g).save(os.path.join(SAVE_RENDER_DIR, f"{safe_name}_view{i}_gt.png"))
                Image.fromarray(p).save(os.path.join(SAVE_RENDER_DIR, f"{safe_name}_view{i}_pred.png"))

        psnrs, ssims, lpipss = [], [], []
        for g, p in zip(gt_views, pred_views):
            psnr, ssim, lp = compute_view_metrics(g, p, loss_fn, DEVICE)
            psnrs.append(psnr)
            ssims.append(ssim)
            lpipss.append(lp)

        row["psnr"] = float(np.mean(psnrs))
        row["ssim"] = float(np.mean(ssims))
        row["lpips"] = float(np.mean(lpipss))

        print(f"  [ok] {tag}: PSNR={row['psnr']:.2f}dB  SSIM={row['ssim']:.3f}  "
              f"LPIPS={row['lpips']:.3f}  (icp_fitness={fitness:.3f})")

    except Exception as e:  # noqa: BLE001
        row["status"] = "error"
        row["error_message"] = f"{e}"
        print(f"  [ERROR] {tag}: {e}")
        traceback.print_exc()

    rows.append(row)

# --- save results ---------------------------------------------------------

df = pd.DataFrame(rows)
per_object_path = os.path.join(OUTPUT_DIR, "texture_metrics_per_object.csv")
df.to_csv(per_object_path, index=False)
print(f"\nPer-object results written to {per_object_path}")

ok_df = df[df["status"] == "ok"]
metric_cols = [c for c in ["psnr", "ssim", "lpips", "icp_fitness", "icp_inlier_rmse"] if c in ok_df.columns]

if len(ok_df):
    category_avg = ok_df.groupby("category")[metric_cols].mean().reset_index()
    category_avg["n_objects"] = ok_df.groupby("category").size().values
    category_avg_path = os.path.join(OUTPUT_DIR, "texture_metrics_per_category_avg.csv")
    category_avg.to_csv(category_avg_path, index=False)
    print(f"Per-category averages written to {category_avg_path}\n")
    print(category_avg.to_string(index=False))
else:
    print("No successful results to average.")

n_ok = (df["status"] == "ok").sum()
n_err = (df["status"] == "error").sum()
print(f"\nDone. ok={n_ok} error={n_err} (of {len(df)} total)")